## inputs
https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3

https://www.kaggle.com/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3

https://www.kaggle.com/datasets/jeroencottaar/taaf-kaggle-source-share

https://www.kaggle.com/datasets/driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot

## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

## 4.1 Conservative runtime monkey patch

This cell leaves the read-only source dataset untouched and patches the imported modules in memory before the benchmark pickle is restored. It fixes only two code-verified issues:

- `ACTION7` is exposed by the official engine and by several official environments, but the bundled model-to-engine mapping cannot convert it back into an executable action. The patch adds a neutral `ACTION7` round trip without guessing its game-specific meaning.
- The framework exposes intermediate animation frames, while the baseline passes only the final frame to the agent. The patch adds compact animation metadata to `last_action_result`; it does not add images or raw frames to the context.

RESET behavior, mouse-coordinate clipping, model settings, randomness, concurrency, budgets, source datasets, and benchmark objects are otherwise unchanged.


In [ ]:
# Baseline-preservation mode.
# Keep the mounted dataset implementation as the only source of solver behavior.
# Intentionally do not monkey-patch action mappings, RESET handling, solver methods,
# tool-result schemas, animation metadata, or model-facing prompts.

BASELINE_RUNTIME_STATUS = {
    "mode": "dataset-baseline",
    "runtime_monkey_patch": False,
    "action_mapping_changed": False,
    "reset_behavior_changed": False,
    "solver_methods_changed": False,
    "tool_result_schema_changed": False,
    "prompt_changed": False,
    "dataset_modified": False,
}

print(
    "taaf.kaggle: runtime patches disabled: "
    f"{BASELINE_RUNTIME_STATUS}"
)

## 4.2 Score-stability rollback

The previous patch intentionally remains in this notebook for traceability, but its score-sensitive animation and prompt changes are neutralized here before the benchmark is restored. This revision retains only the minimum compatibility fix required to convert an already-visible `ACTION7` label back to the engine action.

This cell restores the original solver methods, original compact action-result schema, and original system prompt. It does not change the dataset, model, sampling, concurrency, budgets, reset behavior, mouse handling, notebook metadata, existing cells, or existing outputs.


In [ ]:
# No rollback is required because the preceding runtime-patch cell is intentionally inert.
# Leave all imported modules exactly as provided by the mounted source dataset.

BASELINE_ROLLBACK_STATUS = {
    "rollback_required": False,
    "dataset_baseline_preserved": True,
}

print(
    "taaf.kaggle: baseline modules preserved: "
    f"{BASELINE_ROLLBACK_STATUS}"
)

## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts — the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# Keep the deserialized benchmark and solver exactly as provided by benchmark_initial.pkl.
# No trajectory memory, loop detector, action suppression, prompt hint, budget override,
# model change, sampling change, game reordering, or other score-sensitive customization.

BASELINE_CUSTOMIZATION_STATUS = {
    "benchmark_changed": False,
    "solver_changed": False,
    "local_budget_changed": False,
    "model_facing_state_changed": False,
    "dataset_modified": False,
}

print(
    "taaf.kaggle: customization disabled: "
    f"{BASELINE_CUSTOMIZATION_STATUS}"
)

In [ ]:
# Minimal vLLM health check before benchmark execution
import time
import requests

def wait_vllm_ready(timeout=180):
    start = time.time()

    while time.time() - start < timeout:
        try:
            r = requests.get(
                "http://127.0.0.1:1234/v1/models",
                timeout=5
            )

            if r.status_code == 200:
                print("vLLM server ready")
                return True

        except Exception:
            pass

        time.sleep(5)

    raise RuntimeError(
        "vLLM server is not alive before benchmark start"
    )


wait_vllm_ready()

## 7. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise — an interactive "Save & Run" — play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)

bm.n_passes = 1
bm.game_weights = None

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - min(600.0, budget / 2))

# Play the benchmark; teardown commands run even if the run raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
    if not TRUE_SUBMISSION:
        # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)
finally:
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())

## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")

In [ ]:
# === CLEAR LOCAL SCORE CARD ===
# Keep this as the final cell. It summarizes the completed local/offline run directly from `bm`.
import re
from html import escape

from IPython.display import HTML, display

if TRUE_SUBMISSION:
    print("Official competition rerun: the local estimate card is intentionally hidden.")
elif not getattr(bm, "game_runs", None):
    print("No completed local benchmark data is available yet.")
else:
    from taaf import diagnostics as taaf_diagnostics

    _summary = taaf_diagnostics.run_summary_text(bm)

    def _extract(pattern, cast=str, default=None):
        match = re.search(pattern, _summary, flags=re.MULTILINE)
        if not match:
            return default
        try:
            return cast(match.group(1).strip())
        except Exception:
            return default

    _mean = _extract(r"^mean score:\s*([0-9.]+)\s*$", float, 0.0)
    _median = _extract(r"^median score:\s*([0-9.]+)\s*$", float, 0.0)
    _duration = _extract(r"^duration:\s*(.+?)\s*$", str, "unknown")
    _games = _extract(r"^games:\s*(\d+)\s*$", int, 0)
    _won = _extract(r"^runs:\s*\d+\s*\(won:\s*(\d+)\)\s*$", int, 0)
    _actions = _extract(r"^total actions:\s*(\d+)\s*$", int, 0)
    _tokens = _extract(r"^total tokens:\s*(\d+)\s*$", int, 0)

    _per_game = re.findall(
        r"^\s+\S+:\s+score=([0-9.]+),\s+levels=([0-9.]+)/([0-9.]+),",
        _summary,
        flags=re.MULTILINE,
    )
    _positive = sum(float(score) > 0 for score, _, _ in _per_game)
    _levels_done = sum(float(done) for _, done, _ in _per_game)
    _levels_total = sum(float(total) for _, _, total in _per_game)

    _budget_s = float(globals().get("LOCAL_FAST_EVAL_SECONDS", 0.0) or 0.0)
    _budget_label = (
        f"{_budget_s / 60:.1f} min/game"
        if _budget_s > 0
        else "full local budget"
    )
    _level_label = (
        f"{_levels_done:.0f}/{_levels_total:.0f}"
        if _levels_total > 0
        else "unknown"
    )
    _positive_label = (
        f"{_positive}/{len(_per_game)}"
        if _per_game
        else "unknown"
    )

    display(HTML(f"""
    <div style="border:1px solid #6b7280;border-radius:14px;padding:20px 24px;margin:14px 0;max-width:920px;font-family:Arial,sans-serif">
      <div style="font-size:15px;font-weight:800;letter-spacing:.05em">ARC-AGI-3 LOCAL FAST EVALUATION</div>
      <div style="font-size:46px;font-weight:850;line-height:1.15;margin-top:8px">{_mean:.2f}<span style="font-size:18px;font-weight:500"> / 100</span></div>
      <div style="font-size:14px;margin-top:4px">Estimated mean score on the local public environments (not the official hidden-environment leaderboard score)</div>
      <hr style="margin:16px 0;border:none;border-top:1px solid #6b7280">
      <table style="border-collapse:collapse;width:100%;font-size:14px;line-height:1.9">
        <tr><td>Median score</td><td><b>{_median:.2f}</b></td><td>Games fully solved</td><td><b>{_won}/{_games}</b></td></tr>
        <tr><td>Games with positive score</td><td><b>{_positive_label}</b></td><td>Levels completed</td><td><b>{_level_label}</b></td></tr>
        <tr><td>Total actions</td><td><b>{_actions:,}</b></td><td>Total generated tokens</td><td><b>{_tokens:,}</b></td></tr>
        <tr><td>Benchmark duration</td><td><b>{escape(_duration)}</b></td><td>Local time budget</td><td><b>{escape(_budget_label)}</b></td></tr>
      </table>
      <div style="font-size:12px;margin-top:14px;opacity:.78">Fast evaluation only shortens the local per-game time budget. The gateway, model, sampling settings, concurrency, game list, and submission path remain unchanged for official competition reruns. Scores under the shortened budget are usually lower than results from the full 132-minute-per-game local evaluation.</div>
    </div>
    """))
